In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sb
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, root_mean_squared_error
import joblib
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [5]:
# Data Collection and inital exploration 
data = pd.read_csv("../datasets/house-price/Bangalore.csv")
print(data.head())

      Price  Area                         Location  No. of Bedrooms  Resale  \
0  30000000  3340                 JP Nagar Phase 1                4       0   
1   7888000  1045       Dasarahalli on Tumkur Road                2       0   
2   4866000  1179  Kannur on Thanisandra Main Road                2       0   
3   8358000  1675                     Doddanekundi                3       0   
4   6845000  1670                          Kengeri                3       0   

   MaintenanceStaff  Gymnasium  SwimmingPool  LandscapedGardens  JoggingTrack  \
0                 1          1             1                  1             1   
1                 0          1             1                  1             1   
2                 0          1             1                  1             1   
3                 0          0             0                  0             0   
4                 1          1             1                  1             1   

   ...  LiftAvailable  BED  VaastuComp

In [6]:
# X -> Input features (all columns except SalePrice)
# y -> Target variable (SalePrice)
X = data.drop("Price", axis=1)
y = data["Price"]

# Select numerical columns (int and float types)
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
# Select categorical columns (object/string types)
cat_cols = X.select_dtypes(include=["object"]).columns

# Numerical pipeline:
# - Impute missing values using the median (robust to outliers)
num_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median"))])

# Categorical pipeline:
# - Impute missing values using most frequent category
# - Encode categories using One-Hot Encoding
cat_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

# ==========================================
# Combine Pipelines Using ColumnTransformer
# ==========================================
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            num_pipeline,
            num_cols,
        ),  # Apply numerical pipeline to numerical columns
        (
            "cat",
            cat_pipeline,
            cat_cols,
        ),  # Apply categorical pipeline to categorical columns
    ]
)

# Create Final Machine Learning Pipeline
mlr = Pipeline(
    steps=[
        ("preprocess", preprocessor),  # Data preprocessing step
        ("model", LinearRegression()),  # Multiple Linear Regression model
    ]
)
# =========================
# Train-Test Split
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,  # 20% data for testing
    random_state=42,  # Ensures reproducibility
)

# Model Training
mlr.fit(X_train, y_train)

# Model Evaluation
y_pred = mlr.predict(X_test)
print("MLR R2:", r2_score(y_test, y_pred))
print("MLR MAE:", mean_absolute_error(y_test, y_pred))
print("MLR MSE:", mean_squared_error(y_test, y_pred))
print("MLR RMSE:", root_mean_squared_error(y_test, y_pred))

joblib.dump(mlr, "model_multiple_linear_regression.pkl")
print("Model saved as model_multiple_linear_regression.pkl")

MLR R2: 0.1813893266864951
MLR MAE: 5055098.995611279
MLR MSE: 219712712054957.97
MLR RMSE: 14822709.335845387
Model saved as model_multiple_linear_regression.pkl
